# Block 3 — Project Machine Learning for Gas Adsorption

**Group G — Team members**

- Marie Lacroix
- Chloé Baruselli
- Bertille Delloye
- Enéa Drezet--Marçot


## Import packages we will need

In [ ]:
# Install dependencies
%pip install -r requirements.txt

# basics
import os
import numpy as np
import pprint as pp

# pandas is used to read/process data
import pandas as pd

# machine learning dependencies
# scaling of data
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
# train/test split
from sklearn.model_selection import train_test_split
# model selection
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
# the KRR model
from sklearn.kernel_ridge import KernelRidge
# linear model
from sklearn.linear_model import LinearRegression
# pipeline to streamline modeling pipelines
from sklearn.pipeline import Pipeline
# principal component analysis
from sklearn.decomposition import PCA
# polynomial kernel
from sklearn.metrics.pairwise import polynomial_kernel
# Dummy model as baseline
from sklearn.dummy import DummyClassifier, DummyRegressor
# Variance Threshold for feature selection
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
# metrics to measure model performance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             mean_absolute_error, mean_squared_error, max_error)

# save/load models
import joblib

# For the permutation importance implementation
from joblib import Parallel
from joblib import delayed
from sklearn.metrics import check_scoring
from sklearn.utils import Bunch
from sklearn.utils import check_random_state
from sklearn.utils import check_array

# plotting
import matplotlib.pyplot as plt
%matplotlib inline
from pymatviz.scatter import density_scatter_with_hist as hist_density


RANDOM_SEED = 4242424242
DATA_DIR = 'data'
DATA_FILE = os.path.join(DATA_DIR, 'data.csv')

np.random.seed(RANDOM_SEED)


In [ ]:

other_descriptors = ["CellV [A^3]"]

geometric_descriptors = [
    "Di",
    "Df",
    "Dif",
    "density [g/cm^3]",
    "total_SA_volumetric",
    "total_SA_gravimetric",
    "total_POV_volumetric",
    "total_POV_gravimetric",
]

linker_descriptors = [
    "f-lig-chi-0",
    "f-lig-chi-1",
    "f-lig-chi-2",
    "f-lig-chi-3",
    "f-lig-Z-0",
    "f-lig-Z-1",
    "f-lig-Z-2",
    "f-lig-Z-3",
    "f-lig-I-0",
    "f-lig-I-1",
    "f-lig-I-2",
    "f-lig-I-3",
    "f-lig-T-0",
    "f-lig-T-1",
    "f-lig-T-2",
    "f-lig-T-3",
    "f-lig-S-0",
    "f-lig-S-1",
    "f-lig-S-2",
    "f-lig-S-3",
    "lc-chi-0-all",
    "lc-chi-1-all",
    "lc-chi-2-all",
    "lc-chi-3-all",
    "lc-Z-0-all",
    "lc-Z-1-all",
    "lc-Z-2-all",
    "lc-Z-3-all",
    "lc-I-0-all",
    "lc-I-1-all",
    "lc-I-2-all",
    "lc-I-3-all",
    "lc-T-0-all",
    "lc-T-1-all",
    "lc-T-2-all",
    "lc-T-3-all",
    "lc-S-0-all",
    "lc-S-1-all",
    "lc-S-2-all",
    "lc-S-3-all",
    "lc-alpha-0-all",
    "lc-alpha-1-all",
    "lc-alpha-2-all",
    "lc-alpha-3-all",
    "D_lc-chi-0-all",
    "D_lc-chi-1-all",
    "D_lc-chi-2-all",
    "D_lc-chi-3-all",
    "D_lc-Z-0-all",
    "D_lc-Z-1-all",
    "D_lc-Z-2-all",
    "D_lc-Z-3-all",
    "D_lc-I-0-all",
    "D_lc-I-1-all",
    "D_lc-I-2-all",
    "D_lc-I-3-all",
    "D_lc-T-0-all",
    "D_lc-T-1-all",
    "D_lc-T-2-all",
    "D_lc-T-3-all",
    "D_lc-S-0-all",
    "D_lc-S-1-all",
    "D_lc-S-2-all",
    "D_lc-S-3-all",
    "D_lc-alpha-0-all",
    "D_lc-alpha-1-all",
    "D_lc-alpha-2-all",
    "D_lc-alpha-3-all",
]

metalcenter_descriptors = [
    "mc_CRY-chi-0-all",
    "mc_CRY-chi-1-all",
    "mc_CRY-chi-2-all",
    "mc_CRY-chi-3-all",
    "mc_CRY-Z-0-all",
    "mc_CRY-Z-1-all",
    "mc_CRY-Z-2-all",
    "mc_CRY-Z-3-all",
    "mc_CRY-I-0-all",
    "mc_CRY-I-1-all",
    "mc_CRY-I-2-all",
    "mc_CRY-I-3-all",
    "mc_CRY-T-0-all",
    "mc_CRY-T-1-all",
    "mc_CRY-T-2-all",
    "mc_CRY-T-3-all",
    "mc_CRY-S-0-all",
    "mc_CRY-S-1-all",
    "mc_CRY-S-2-all",
    "mc_CRY-S-3-all",
    "D_mc_CRY-chi-0-all",
    "D_mc_CRY-chi-1-all",
    "D_mc_CRY-chi-2-all",
    "D_mc_CRY-chi-3-all",
    "D_mc_CRY-Z-0-all",
    "D_mc_CRY-Z-1-all",
    "D_mc_CRY-Z-2-all",
    "D_mc_CRY-Z-3-all",
    "D_mc_CRY-I-0-all",
    "D_mc_CRY-I-1-all",
    "D_mc_CRY-I-2-all",
    "D_mc_CRY-I-3-all",
    "D_mc_CRY-T-0-all",
    "D_mc_CRY-T-1-all",
    "D_mc_CRY-T-2-all",
    "D_mc_CRY-T-3-all",
    "D_mc_CRY-S-0-all",
    "D_mc_CRY-S-1-all",
    "D_mc_CRY-S-2-all",
    "D_mc_CRY-S-3-all",
]

functionalgroup_descriptors = [
    "func-chi-0-all",
    "func-chi-1-all",
    "func-chi-2-all",
    "func-chi-3-all",
    "func-Z-0-all",
    "func-Z-1-all",
    "func-Z-2-all",
    "func-Z-3-all",
    "func-I-0-all",
    "func-I-1-all",
    "func-I-2-all",
    "func-I-3-all",
    "func-T-0-all",
    "func-T-1-all",
    "func-T-2-all",
    "func-T-3-all",
    "func-S-0-all",
    "func-S-1-all",
    "func-S-2-all",
    "func-S-3-all",
    "func-alpha-0-all",
    "func-alpha-1-all",
    "func-alpha-2-all",
    "func-alpha-3-all",
    "D_func-chi-0-all",
    "D_func-chi-1-all",
    "D_func-chi-2-all",
    "D_func-chi-3-all",
    "D_func-Z-0-all",
    "D_func-Z-1-all",
    "D_func-Z-2-all",
    "D_func-Z-3-all",
    "D_func-I-0-all",
    "D_func-I-1-all",
    "D_func-I-2-all",
    "D_func-I-3-all",
    "D_func-T-0-all",
    "D_func-T-1-all",
    "D_func-T-2-all",
    "D_func-T-3-all",
    "D_func-S-0-all",
    "D_func-S-1-all",
    "D_func-S-2-all",
    "D_func-S-3-all",
    "D_func-alpha-0-all",
    "D_func-alpha-1-all",
    "D_func-alpha-2-all",
    "D_func-alpha-3-all",
]


summed_linker_descriptors = [
    "sum-f-lig-chi-0",
    "sum-f-lig-chi-1",
    "sum-f-lig-chi-2",
    "sum-f-lig-chi-3",
    "sum-f-lig-Z-0",
    "sum-f-lig-Z-1",
    "sum-f-lig-Z-2",
    "sum-f-lig-Z-3",
    "sum-f-lig-I-0",
    "sum-f-lig-I-1",
    "sum-f-lig-I-2",
    "sum-f-lig-I-3",
    "sum-f-lig-T-0",
    "sum-f-lig-T-1",
    "sum-f-lig-T-2",
    "sum-f-lig-T-3",
    "sum-f-lig-S-0",
    "sum-f-lig-S-1",
    "sum-f-lig-S-2",
    "sum-f-lig-S-3",
    "sum-lc-chi-0-all",
    "sum-lc-chi-1-all",
    "sum-lc-chi-2-all",
    "sum-lc-chi-3-all",
    "sum-lc-Z-0-all",
    "sum-lc-Z-1-all",
    "sum-lc-Z-2-all",
    "sum-lc-Z-3-all",
    "sum-lc-I-0-all",
    "sum-lc-I-1-all",
    "sum-lc-I-2-all",
    "sum-lc-I-3-all",
    "sum-lc-T-0-all",
    "sum-lc-T-1-all",
    "sum-lc-T-2-all",
    "sum-lc-T-3-all",
    "sum-lc-S-0-all",
    "sum-lc-S-1-all",
    "sum-lc-S-2-all",
    "sum-lc-S-3-all",
    "sum-lc-alpha-0-all",
    "sum-lc-alpha-1-all",
    "sum-lc-alpha-2-all",
    "sum-lc-alpha-3-all",
    "sum-D_lc-chi-0-all",
    "sum-D_lc-chi-1-all",
    "sum-D_lc-chi-2-all",
    "sum-D_lc-chi-3-all",
    "sum-D_lc-Z-0-all",
    "sum-D_lc-Z-1-all",
    "sum-D_lc-Z-2-all",
    "sum-D_lc-Z-3-all",
    "sum-D_lc-I-0-all",
    "sum-D_lc-I-1-all",
    "sum-D_lc-I-2-all",
    "sum-D_lc-I-3-all",
    "sum-D_lc-T-0-all",
    "sum-D_lc-T-1-all",
    "sum-D_lc-T-2-all",
    "sum-D_lc-T-3-all",
    "sum-D_lc-S-0-all",
    "sum-D_lc-S-1-all",
    "sum-D_lc-S-2-all",
    "sum-D_lc-S-3-all",
    "sum-D_lc-alpha-0-all",
    "sum-D_lc-alpha-1-all",
    "sum-D_lc-alpha-2-all",
    "sum-D_lc-alpha-3-all",
]

summed_metalcenter_descriptors = [
    "sum-mc_CRY-chi-0-all",
    "sum-mc_CRY-chi-1-all",
    "sum-mc_CRY-chi-2-all",
    "sum-mc_CRY-chi-3-all",
    "sum-mc_CRY-Z-0-all",
    "sum-mc_CRY-Z-1-all",
    "sum-mc_CRY-Z-2-all",
    "sum-mc_CRY-Z-3-all",
    "sum-mc_CRY-I-0-all",
    "sum-mc_CRY-I-1-all",
    "sum-mc_CRY-I-2-all",
    "sum-mc_CRY-I-3-all",
    "sum-mc_CRY-T-0-all",
    "sum-mc_CRY-T-1-all",
    "sum-mc_CRY-T-2-all",
    "sum-mc_CRY-T-3-all",
    "sum-mc_CRY-S-0-all",
    "sum-mc_CRY-S-1-all",
    "sum-mc_CRY-S-2-all",
    "sum-mc_CRY-S-3-all",
    "sum-D_mc_CRY-chi-0-all",
    "sum-D_mc_CRY-chi-1-all",
    "sum-D_mc_CRY-chi-2-all",
    "sum-D_mc_CRY-chi-3-all",
    "sum-D_mc_CRY-Z-0-all",
    "sum-D_mc_CRY-Z-1-all",
    "sum-D_mc_CRY-Z-2-all",
    "sum-D_mc_CRY-Z-3-all",
    "sum-D_mc_CRY-I-0-all",
    "sum-D_mc_CRY-I-1-all",
    "sum-D_mc_CRY-I-2-all",
    "sum-D_mc_CRY-I-3-all",
    "sum-D_mc_CRY-T-0-all",
    "sum-D_mc_CRY-T-1-all",
    "sum-D_mc_CRY-T-2-all",
    "sum-D_mc_CRY-T-3-all",
    "sum-D_mc_CRY-S-0-all",
    "sum-D_mc_CRY-S-1-all",
    "sum-D_mc_CRY-S-2-all",
    "sum-D_mc_CRY-S-3-all",
]

summed_functionalgroup_descriptors = [
    "sum-func-chi-0-all",
    "sum-func-chi-1-all",
    "sum-func-chi-2-all",
    "sum-func-chi-3-all",
    "sum-func-Z-0-all",
    "sum-func-Z-1-all",
    "sum-func-Z-2-all",
    "sum-func-Z-3-all",
    "sum-func-I-0-all",
    "sum-func-I-1-all",
    "sum-func-I-2-all",
    "sum-func-I-3-all",
    "sum-func-T-0-all",
    "sum-func-T-1-all",
    "sum-func-T-2-all",
    "sum-func-T-3-all",
    "sum-func-S-0-all",
    "sum-func-S-1-all",
    "sum-func-S-2-all",
    "sum-func-S-3-all",
    "sum-func-alpha-0-all",
    "sum-func-alpha-1-all",
    "sum-func-alpha-2-all",
    "sum-func-alpha-3-all",
    "sum-D_func-chi-0-all",
    "sum-D_func-chi-1-all",
    "sum-D_func-chi-2-all",
    "sum-D_func-chi-3-all",
    "sum-D_func-Z-0-all",
    "sum-D_func-Z-1-all",
    "sum-D_func-Z-2-all",
    "sum-D_func-Z-3-all",
    "sum-D_func-I-0-all",
    "sum-D_func-I-1-all",
    "sum-D_func-I-2-all",
    "sum-D_func-I-3-all",
    "sum-D_func-T-0-all",
    "sum-D_func-T-1-all",
    "sum-D_func-T-2-all",
    "sum-D_func-T-3-all",
    "sum-D_func-S-0-all",
    "sum-D_func-S-1-all",
    "sum-D_func-S-2-all",
    "sum-D_func-S-3-all",
    "sum-D_func-alpha-0-all",
    "sum-D_func-alpha-1-all",
    "sum-D_func-alpha-2-all",
    "sum-D_func-alpha-3-all",
]


 $\color{DarkBlue}{\textsf{Short question}}$
- We declared a global variable to fix the random seed (`RANDOM_SEED`). Why did we do this?  

A fixed random seed is used to make the results reproducible. Random operations performed during the workflow generate the same sequence of pseudo-random values each time the notebook is run, allowing the same data splits and results to be reproduced.


## Hands-on Project: Carbon-dioxide uptake in MOFs

In this exercise we will build a model that can predict the CO₂ uptake of metal-organic frameworks (MOFs), which are crystalline materials consisting of inorganic metal nodes linked by organic linkers.

![MOF building principle](assets/mof_building_principle.png)

There are two main **learning goals** for this exercise: 

1. Understand the typical workflow for machine learning in materials science. We will cover exploratory data analysis (EDA) and supervised learning (KRR).

2. Get familiar with some Python packages that are useful for data analysis and visualization. 

At the end of the exercise, you will produce an interactive plot like the one below, comparing the predictions of your model against CO₂ computed with GCMC simulations.
The histograms show the distributions of the errors on the training set (left) and on the test set (right).



<img src="assets/result.gif" alt="Parity interactive" width="700"/>

This exercise requires a basic knowledge of Python, e.g. that you can write list comprehensions, and are able to read documentation of functions provided by Python packages.
You will be asked to provide some function arguments (indicated by `#fillme` comments).

You can execute all the following code cells by pressing SHIFT and ENTER and get informations about the functions by pressing TAB when you are between the parentheses (see the notes for more tips). 

Also the [sklearn documentation](https://scikit-learn.org/stable/user_guide.html) is a great source of reference with many explanations and examples.

In pandas dataframe (df) you can select columns using their name by running `df[columnname]`. If at any point you think that the dataset is too large for your computer, you can select a subset using `df.sample()` or by making the test set larger in the train/test split (section 2). 

## 1. Import the data

In [ ]:
df = pd.read_csv(DATA_FILE)

Let's take a look at the first few rows to see if everythings seems reasonable ...

In [ ]:
df.head()

<details>
<summary> <font color='green'>Click here for a hint</font></summary>
<ul>
    <li>Use something like <code>pd.options.display.max_columns=100</code> to adjust how many columns are shown.<code>pd.options.display.max_columns=100</code>  would show at maximum 100 columns. </li>
</ul>
</details>

Let's also get some basic information ...

In [ ]:
df.info()

 $\color{DarkBlue}{\textsf{Short question}}$
- How many materials are in the dataset?

There are **17,379 materials** in the dataset, corresponding to one MOF per row.

- Which datatypes do we deal with?

The dataset contains **342 `float64` columns** and **1 `object` column**, which contains the MOF names.


Below, we define three global variables (hence upper case), which are the *names* of our feature and target columns. We will use the `TARGET` for the actual regression and the `TARGET_BINARY` only for the stratified train/test split. The `FEATURES` variable is a list of column names of our dataframe.

In [ ]:
TARGET = "pure_uptake_CO2_298.00_1600000"
TARGET_BINARY = "target_binned"  # will be created later
FEATURES = (
    geometric_descriptors
    + summed_functionalgroup_descriptors
    + summed_linker_descriptors
    + summed_metalcenter_descriptors
)


As descriptors we will use geometric properties such as density, pore volume, etc. and [revised autocorrelation functions](https://pubs.acs.org/doi/abs/10.1021/acs.jpca.7b08750) (RACs) that have been optimized for describing inorganic compounds ([and recently adapated for MOFs](https://www.nature.com/articles/s41467-020-17755-8))

Examples for pore geometry descriptors (in `geometric_descriptors`) include: $D_i$ (the size of the largest included sphere), $D_f$ (the largest free sphere), and $D_{if}$ (the largest included free sphere) along the pore $-$ three ways of characterizing pore size. 

![pore diameters](assets/spheres.png)

Also included are the surface area (SA) of the pore, and the probe-occupiable pore volume (POV).
More details on the description of pore geometries can be found in [Ongari et al.](https://pubs.acs.org/doi/abs/10.1021/acs.langmuir.7b01682)

RACs (in the lists starting with `summed_...`) operate on the structure graph and encode information about the metal center, linkers and the functional groups as differences or products of heuristics that are relevant for inorganic chemistry, such as electronegativity ($\chi$), connectivity ($T$), identity ($I$), covalent radii ($S$), and nuclear charge ($Z$).


<img src="assets/racs.png" alt="RACs scheme from the lecture" width="700"/>

The number in the descriptornames shows the coordination shell that was considered in the calculation of the RACs.

The target we use for this application is the high-pressure CO₂ uptake. This is the amount of CO₂ (mmol) the MOF can load per gram.

## 2. Split the data

Next, we split our data into a training set and a test set.

In order to prevent *any* information of the test set from leaking into our model, we split *before* starting to analyze or transform our data. For more details on why this matters, see [chapter 7.10.2 of Elements of Statistical Learning](https://web.stanford.edu/~hastie/ElemStatLearn//printings/ESLII_print10.pdf).

### 2.1. Split with stratification

[Stratification](https://en.wikipedia.org/wiki/Stratified_sampling) ensures that the class distributions (ratio of "good" to "bad" materials) are the same in the training and test set.

 $\color{DarkBlue}{\textsf{Short question}}$

- Why is this important? What could happen if we would not do this? 

First, we need two fundamentally different datasets : a training set to build the model and a test set to evaluate the final model on unseen data. But, the whole dataset can not be separated blindly, because both sets must represent the same populations. Therefore, we use stratification to ensure that both training and test sets contain comparable proportions of low- and high-performing MOFs.
 

For stratification to work, we to define what makes a "good" or a "bad" material. We will use 15 mmol CO₂ / g as the threshold for the uptake, thus binarizing our continuous target variable. (You can choose it based on the histogram of the variables).

 $\color{DarkBlue}{\textsf{Short Exercise}}$
 - add a column 'target_binary' that encodes whether a material is low performing (`0`) or high perfoming (`1`) by comparing the uptake with the `THRESHOLD`

<details>
<summary> <font color='green'>Click here for a hint</font></summary>
<ul>
    <li> you can use <a href='https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.cut.html'>pd.cut</a>, 
    <a href='https://stackoverflow.com/questions/4406389/if-else-in-a-list-comprehension'>list comprehension</a>, the <a href='https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.Binarizer.html#sklearn.preprocessing.Binarizer'> binarizer in sklearn </a>...) </li>
    <li> a list comprehension example: <code> [1 if value > THRESHOLD else 0 for value in df[TARGET]] </code> </li>
</ul>
</details>

In [ ]:
THRESHOLD = 15  # in units of mmol CO2/g
df[TARGET_BINARY] = [
    1 if uptake > THRESHOLD else 0
    for uptake in df[TARGET]
]

Now, we can perform the actual split into training and test set.

 $\color{DarkBlue}{\textsf{Short Exercise}}$
- select reasonable values for `XX` and `XY` and then perform the test/train splits. What do you consider when making this decision (think about what you would do with really small and really big datasets, what happens if you have only one test point, what happens to the model performance if you have more test points than training points)? 

We selected a 70/30 train-test split. The training set must be sufficiently large for the model to learn the underlying relationships, while the test set must contain enough samples to provide a reliable estimate of model performance. A test set containing only a few points would give a high-variance estimate of the error, whereas allocating too much data to the test set would reduce the amount of data available for training.

- why do we need to perform the split into a training and test set? 

The split is required to evaluate the ability of the model to generalize to unseen materials. The training set is used to fit the model, whereas the test set is kept independent and is only used to estimate the final predictive performance.

- would we use the test set to tune the hyperparameters of our model?

No. The test set should not be used for hyperparameter optimization because this would introduce data leakage and make the final performance estimate overly optimistic. Hyperparameters should instead be selected using a validation set or cross-validation performed on the training data.

<details>
<summary> <font color='green'>Click here for a hint</font></summary>
<ul>
    <li>The `size` arguments can either be integers or, often more convenient, decimals like 0.1</li>
    <li>When you perform the split into training and test set you need to trade-off bias (pessimistic bias due to little training data) and variance (due to little test data) </li>
    <li>A typical split cloud be 70/30, but for huge dataset the test set might be too big and for small datasets the training set might be too small in this way </li>
</ul>
</details>

In [ ]:
df_train_stratified, df_test_stratified = train_test_split(
    df,
    train_size=0.7,
    test_size=0.3,
    random_state=RANDOM_SEED,
    stratify=df[TARGET_BINARY],
)


## 3. Exploratory data analysis (EDA) 

After we have put the test set aside, we can give the training set a closer look.

### 3.1. Correlations

 $\color{DarkBlue}{\textsf{Short Exercise}}$
- Plot some features against the target property and calculate the Pearson and Spearman correlation coefficient (what is the difference between those correlation coefficients?) 

Pearson correlation measures the strength of a **linear** relationship, whereas Spearman correlation is based on ranks and measures a **monotonic** relationship. Spearman can therefore remain high when the relationship is not perfectly linear.

- What are the strongest correlations? Did you expect them? 

The strongest correlations with high-pressure CO₂ uptake are mainly geometrical: pore volume, surface area and pore size are positively correlated with uptake, while density is strongly negatively correlated. This is expected because, at high pressure, the amount of gas that can be stored depends strongly on the accessible pore space.

- What can be a problem when features are correlated?

Strongly correlated features contain redundant information. They increase dimensionality and computational cost without necessarily improving the model, make individual feature importance harder to interpret, and can lead to unstable coefficients in linear models.

- *Optional:* Do they change if you switch from CO₂ to CH₄ uptake as the target instead? Explain your observation.

At high pressure, the strongest correlations remain broadly similar for CH₄: geometrical descriptors such as pore volume, surface area and density remain important because pore filling dominates. The exact correlations change because CH₄ and CO₂ interact differently with the framework. At lower pressure, chemical descriptors are expected to become relatively more important because specific host–guest interactions play a larger role.


To get the correlation matrices, you can use the `df.corr(method=)`method on your dataframe (`df`). You might want to calculate not the full correlation matrix but just the correlation of the features with the targets

<details>
<summary> <font color='green'>Click here for a hint</font></summary>
<ul>
    <li> To get the correlation with a target, you can use indexing. E.g. <code>df.corr(method='spearman')[TARGET]</code></li>
    <li> use <code>.sort_values()</code> method on the output of `df.corr()` to sort by the value of the correlation coefficient  </li>
      <li> You can use something like <code>scatter = hv.Scatter(df, 'Di', [TARGET,  'density [g/cm^3]']).opts(color='density [g/cm^3]', cmap='rainbow')</code> for plotting. Also consider the <a href="https://holoviews.org/reference/elements/matplotlib/Scatter.html"> <code>holoviews</code> documentation</a>. In case <code>holoviews</code> is too new for you, you can of course just use <code>matplotlib</code> and something like <code>plt.scatter(x,y)</code> </li>
</ul>
</details>

In [ ]:
# Pearson correlation
pearson_corr = (
    df_train_stratified[FEATURES + [TARGET]]
    .corr(method="pearson")[TARGET]
    .drop(TARGET)
    .sort_values(ascending=False)
)

# Spearman correlation
spearman_corr = (
    df_train_stratified[FEATURES + [TARGET]]
    .corr(method="spearman")[TARGET]
    .drop(TARGET)
    .sort_values(ascending=False)
)

print("Pearson correlations:")
display(pearson_corr)

print("Spearman correlations:")
display(spearman_corr)

In [ ]:
features_to_plot = [
    "Di",
    "density [g/cm^3]",
    "total_POV_gravimetric",
    "total_SA_gravimetric"
]

for feature in features_to_plot:
    plt.figure(figsize=(6, 4))

    plt.scatter(
        df_train_stratified[feature],
        df_train_stratified[TARGET],
        alpha=0.3
    )

    plt.xlabel(feature)
    plt.ylabel(TARGET)
    plt.title(f"{feature} vs CO2 uptake")

    plt.show()

In [ ]:
spearman_corr_abs = (
    df_train_stratified[FEATURES + [TARGET]]
    .corr(method="spearman")[TARGET]
    .drop(TARGET)
    .sort_values(key=abs, ascending=False)
)

display(spearman_corr_abs.head(10))

## 4. Baselines

For machine learning, it is important to have some *baselines* to which one then compares the results of a model. Think of a classification model for some rare disease where we only have 1% postives. A classification model that only predictes the negatives *all the time* will still have a amazingly high accuracy. To be able to understand if our model is really better than such a simple prediction we need to make the simple prediction first. This is what we call a baseline.

A baseline could be a really simple model, a basic heuristic or the current state of the art.
this. We will use a heuristic.

For this we use sklearn `Dummy` objects that simply calculate the mean, the median or the most frequent case of the training set, when you run the `fit()` method on them (which takes the features matrix $\mathbf{X}$ and the labels $\mathbf{y}$ as arguments.
This is, the prediction of a `DummyRegressor` with `mean` strategy will always be the mean, independent of the input (it will not look at the feature matrix!). 

Instead of using those `sklearn` objects you could also just manually compute the mean or median of the dataset. But we will use those objects as we can learn in this way how to use estimators in `sklearn` and it also allows you to test your full pipeline with different (baseline) models. 
What does this mean? In practice this means that you can use all the regression and classification models shown in the figure below in the same way, they will all have a `fit()` method that accepts `X` and `y` and a predict method that accepts `X` and returns the predictions. 


<img src="https://scikit-learn.org/1.3/_static/ml_map.png" alt="ML Map" width="800"/>

The estimator objects can be always used in the same way 

<img src="https://static.packt-cdn.com/products/9781789800265/graphics/d49a2e95-8f22-42ed-89f1-474b3d028787.png" alt="ML Map" width="400"/>

Using these objects, instead of the mean directly, allows you to easily swap them with other models in pipelines, where one chains many data transformation steps (see section 6).

### 4.1. Build dummy models

$\color{DarkBlue}{\textsf{Short Question}}$
- If you call `.fit(X, y)` on a `DummyRegressor` does it actually use the `X`? If not, why is there still the place for the `X` in the function? If yes, how does it use it?

A `DummyRegressor` does not actually use the features in `X` to learn a relationship between the input variables and the target. For example, with `strategy="mean"`, it simply calculates the mean of the target values in the training set and predicts this same value for every sample.

The reason `X` is still included in `.fit(X, y)` is that Scikit-learn uses the same interface for all regression models. This makes the `DummyRegressor` compatible with the same workflows and pipelines as more complex models, even though it ignores the feature values themselves.

$\color{DarkBlue}{\textsf{Short Exercise}}$
- Create [`DummyRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html) instances for  `mean`, `median`. (e.g. `dummyinstance = DummyRegressor(strategy='mean')`)
- Train them on the training data (`dummyinstance.fit(df_train[FEATURES], df_train[TARGET])`)

<details>
<summary> <font color='green'>Click here for hints</font></summary>
<ul>
    <li> to create <code>DummyRegressor</code> you can for example use <code> dummyregressor_mean = DummyRegressor(strategy='mean') </code> </li>
    <li> to see the implementation of the <code>DummyRegressor</code> you can check out <a href="https://github.com/scikit-learn/scikit-learn/blob/73732e5a0bc9b72c7049dc699d69aaedbb70ef0a/sklearn/dummy.py#L391"> the source code on GitHub</a> </li>
</ul>
</details>

In [ ]:
# Build DummyRegressors
dummyregressor_mean = DummyRegressor(strategy='mean')
dummyregressor_median = DummyRegressor(strategy='median')

In [ ]:
# Fit Dummy Regressors
dummyregressor_mean.fit(
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

dummyregressor_median.fit(
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

#### Evaluate the performance of the dummy models

 $\color{DarkBlue}{\textsf{Short Exercise}}$
- Calculate maximum error, mean absolute error and mean square error for the dummy regressors on training and test set. What would you expect those numbers to be?

Since the dummy models predict the same value for every material and do not use any information from the features, we expect relatively large errors. We also expect the mean regressor to perform better in terms of MSE, since the mean minimizes squared errors, while the median regressor should perform better in terms of MAE, since the median minimizes absolute errors.

- Do the actual values surprise you? 

The results are not surprising. The median regressor gives a slightly lower MAE, while the mean regressor gives a lower MSE, as expected from the definition of these two metrics. The training and test errors are also quite similar, including the maximum errors, which suggests that the two sets have comparable target distributions.

- What does this mean in practice for reporting of metrics/the reasoning behind using baseline models ?

Baseline models give us a reference against which we can compare more complex models. A metric by itself can be difficult to interpret: for example, an MAE of 5 may seem reasonable, but not if a simple model that always predicts the mean already achieves an MAE close to 5. A useful machine-learning model should therefore clearly outperform these simple baselines, showing that it actually learns useful information from the features.

It can be handy to store our metrics of choice in a nested dictionary ([Python dictionaries are key-value pairs](https://www.tutorialspoint.com/python/python_dictionary.htm)): 

```python
{
    'dummyestimator1': {
                        'metric_a_key': metric_a_value, 
                        'metric_b_key': metric_b_value
                    },
    'dummyestimator2': {
                        'metric_a_key': metric_a_value, 
                        'metric_b_key': metric_b_value
                    },
 }
``` 

You will now write functions `get_regression_metrics(model, X, y_true)` that compute the metrics and return this dictionary for a given model. The `predict` method takes the feature matrix $\mathbf{X}$ as input.

In them, we calculate 

$\mathrm {MAE} =\frac{\sum _{i=1}^{n}\left|Y_{i}-\hat{y}_{i}\right|}{n}.$

and 

$\mathrm {MSE} = {\frac {1}{n}}\sum _{i=1}^{n}(Y_{i}-{\hat {Y_{i}}})^{2}.$ 

where $\hat{y}$ are the predictions and, $Y_{i}$ the true values.

as well as the maximum error.

<details>
<summary> <font color='green'>Click here for hints</font></summary>
<ul>
    <li> to perform a prediction using a estimator object, you can call <code> classifier.predict(X) </code> </li>
    <li> to calculate metrics, you can for example call <code>accuracy_score(true_values, predicted_values) </code> </li>
</ul>
</details>

In [ ]:
def get_regression_metrics(model, X, y_true):
    """
    Get a dictionary with regression metrics:

    model: sklearn model with predict method
    X: feature matrix
    y_true: ground truth labels
    """

    y_predicted = model.predict(X)

    mae = mean_absolute_error(y_true, y_predicted)
    mse = mean_squared_error(y_true, y_predicted)
    maximum_error = max_error(y_true, y_predicted)

    metrics_dict = {
        'mae': mae,
        'mse': mse,
        'max_error': maximum_error
    }

    return metrics_dict

In [ ]:
dummy_regressors = [
    ('mean', dummyregressor_mean),
    ('median', dummyregressor_median)
]

In [ ]:
dummy_regressor_results_test = {}
dummy_regressor_results_train = {}

for regressorname, regressor in dummy_regressors:
    print(f"Calculating metrics for {regressorname}")

    dummy_regressor_results_test[regressorname] = get_regression_metrics(
        regressor,
        df_test_stratified[FEATURES],
        df_test_stratified[TARGET]
    )

    dummy_regressor_results_train[regressorname] = get_regression_metrics(
        regressor,
        df_train_stratified[FEATURES],
        df_train_stratified[TARGET]
    )

print("Training results:")
display(dummy_regressor_results_train)

print("Test results:")
display(dummy_regressor_results_test)

## 5. Build actual regression models

Let's build a simple [kernel ridge regression (KRR)](https://emtiyaz.github.io/pcml15/kernel-ridge-regression.pdf) machine learning model and train it with our raw data.
You can try different kernels, but we recommend to start with the Gaussian radial basis function ('rbf') kernel.
 
 $\color{DarkBlue}{\textsf{Short Question}}$
- Do you expect this model to perform better than the dummy models?

Yes, we expect the KRR model to perform better than the dummy regressors because it actually uses the MOF descriptors to predict the CO₂ uptake. Since several of the features showed strong correlations with the target, the model should be able to learn useful relationships from the data instead of predicting the same value for every material.

- Train it and then calculate the performance metrics on the training and test set. How do they compare to the performance of the dummy models?

We expected KRR to perform better than the dummy models because it uses the MOF descriptors to make its predictions. However, the results show that the KRR only performs slightly better on the training set, while its performance on the test set is much worse. The test MAE increases from about 6.3 mmol/g for the dummy model to about 11.7 mmol/g for KRR.

This shows that the current KRR model does not generalize well to unseen materials. A likely reason is that it is trained directly on the raw features, which have very different numerical scales. The next steps, especially feature scaling and hyperparameter optimization, should help improve its performance.

- What is the shape of the Kernel and of the weights? (you can check your answer by looking at the `dual_coef_` attribute of the KRR instance. You can get shapes of objects using the `shape` atrribute 

With \(N\) training samples, the kernel matrix has shape \(N \times N\), because it contains the similarity between every pair of training materials. Since our 70/30 split gives about 12,165 training samples, the kernel matrix has shape \((12165, 12165)\). The KRR model also has one dual coefficient per training sample, so `dual_coef_` has shape \((12165,)\).

In [ ]:
# Train the model with a Gaussian kernel
krr = KernelRidge(kernel='rbf')

krr.fit(
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

In [ ]:
# Get the metrics on the train and test sets
krr_results_train = get_regression_metrics(
    krr,
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

krr_results_test = get_regression_metrics(
    krr,
    df_test_stratified[FEATURES],
    df_test_stratified[TARGET]
)

print("KRR - train:")
print(krr_results_train)

print("KRR - test:")
print(krr_results_test)

print("\nDummy regressors - test:")
print(dummy_regressor_results_test)


In [ ]:
# Check the number of KRR dual coefficients
print("Number of training samples:", len(df_train_stratified))
print("Shape of dual coefficients:", krr.dual_coef_.shape)


## 6. Evaluate the model performance in detail

We have trained our first machine learning model!
We'll first have a closer look at its performance, before learning how to improve it.

 $\color{DarkBlue}{\textsf{Short Exercise}}$
- Create a parity plot (true values against predictions)for the training and test data
- Plot a histogram of the distribution of the training and test errors on the training and test set. Plot the errors also as a function of the true value
- Let's assume we would like to use our model for pre-screening a library of millions of porous materials to zoom-in on those with the most promising gas uptake. Could you tolerate the errors of your model?
- Compare the parity plots for this model with the ones for the dummy models. 
Use the plotting functions below the evaluate all the following models you train.

For this exercise, it can be handy to save the results in a dictionary, e.g. 
```(python)
res_train = {
    'y true': [],
    'y pred': []
}
```

<details>
<summary> <font color='green'>Click here for hints for plotting</font></summary>
<ul>
    <li> If you want to use matplotlib to make the parity plots, you can use the <a href="https://matplotlib.org/3.1.1/api/_as_gen/matplotlib.pyplot.hist2d.html">hist2d function</a> </li>
    <li> To create the frequencies and the edges of a histogram, one can use <code>np.histogram</code></li>
</ul>
</details>

In [ ]:
# Create dictionaries with training and test results

res_train = {
    'y true': df_train_stratified[TARGET],
    'y pred': krr.predict(df_train_stratified[FEATURES])
}

res_test = {
    'y true': df_test_stratified[TARGET],
    'y pred': krr.predict(df_test_stratified[FEATURES])
}

Now, lets calculate the errors

In [ ]:
res_train["error"] = res_train["y true"] - res_train["y pred"]
res_test["error"] = res_test["y true"] - res_test["y pred"]

Now, plot the parity plots and error distributions

<details>
<summary> <font color='green'>Click here for hints for plotting</font></summary>
If you want interactive plots, you can use the following code:
<pre><code>
hv.extension("bokeh")
hex_train = hv.HexTiles(res_train, ["y true", "y pred"]).hist(
    dimension=["y true", "y pred"]
)
hex_test = hv.HexTiles(res_test, ["y true", "y pred"]).hist(
    dimension=["y true", "y pred"]
)
hex_train + hex_test
</code>
</details>

In [ ]:
import holoviews as hv

hv.extension("bokeh")

hex_train = hv.HexTiles(
    res_train,
    ["y true", "y pred"]
).opts(
    title="KRR - Training set",
    xlabel="True CO2 uptake (mmol/g)",
    ylabel="Predicted CO2 uptake (mmol/g)"
).hist(
    dimension=["y true", "y pred"]
)

hex_test = hv.HexTiles(
    res_test,
    ["y true", "y pred"]
).opts(
    title="KRR - Test set",
    xlabel="True CO2 uptake (mmol/g)",
    ylabel="Predicted CO2 uptake (mmol/g)"
).hist(
    dimension=["y true", "y pred"]
)

display(hex_train + hex_test)

In [ ]:
# Parity plots for the dummy baselines on the test set
res_dummy_mean = {
    "y true": df_test_stratified[TARGET],
    "y pred": dummyregressor_mean.predict(df_test_stratified[FEATURES])
}

res_dummy_median = {
    "y true": df_test_stratified[TARGET],
    "y pred": dummyregressor_median.predict(df_test_stratified[FEATURES])
}

hex_dummy_mean = hv.HexTiles(
    res_dummy_mean,
    ["y true", "y pred"]
).opts(
    title="Dummy mean - Test set",
    xlabel="True CO2 uptake (mmol/g)",
    ylabel="Predicted CO2 uptake (mmol/g)"
).hist(dimension=["y true", "y pred"])

hex_dummy_median = hv.HexTiles(
    res_dummy_median,
    ["y true", "y pred"]
).opts(
    title="Dummy median - Test set",
    xlabel="True CO2 uptake (mmol/g)",
    ylabel="Predicted CO2 uptake (mmol/g)"
).hist(dimension=["y true", "y pred"])

display(hex_dummy_mean + hex_dummy_median)


In [ ]:
# Error distributions
plt.figure(figsize=(7, 5))
plt.hist(res_train["error"], bins=50, alpha=0.6, label="Train")
plt.hist(res_test["error"], bins=50, alpha=0.6, label="Test")
plt.xlabel("Prediction error (mmol CO2/g)")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
# Error as a function of the true value
plt.figure(figsize=(7, 5))

plt.scatter(
    res_train["y true"],
    res_train["error"],
    alpha=0.3,
    label="Train"
)

plt.scatter(
    res_test["y true"],
    res_test["error"],
    alpha=0.3,
    label="Test"
)

plt.axhline(0, linestyle="--")
plt.xlabel("True CO2 uptake (mmol/g)")
plt.ylabel("Prediction error (mmol/g)")
plt.legend()
plt.show()

### Interpretation

The raw KRR behaves very differently on the training and test sets. On the training set, the predictions follow the overall trend, whereas on the test set many predictions are close to zero even for materials with high true CO₂ uptake.

The error histogram confirms this behaviour: most errors are positive, meaning that the model systematically **underpredicts** the uptake, and the test errors are much broader than the training errors. The error also increases with the true uptake, which is particularly problematic because high-uptake materials are the ones we want to identify during pre-screening.

With a test MAE of about **11.7 mmol/g** and errors above **30 mmol/g**, the raw KRR is not reliable enough for pre-screening. It even performs worse than the dummy baselines on the test set, showing that model complexity alone does not guarantee good generalization.


## 7. Improve the model 

Our training set still has a couple of issues you might have noticed:
- The feature values are not scaled (different features are measured in different units ...)
- Some features are basically constant, i.e. do not contain relevant information and just increase the dimensionality of the problem 
- Some feature distributions are skewed (which is more relevant for some models than for others ...)

$\color{DarkBlue}{\textsf{Short Question}}$
- Why might the scaling of the features be relevant for a machine learning model? 

Feature scaling is important because the different descriptors can have very different numerical ranges and units. Without scaling, features with larger values can have a much stronger influence on the model simply because of their scale, even if they are not more important physically.

This is especially relevant for KRR with an RBF kernel, since the kernel is based on distances between samples. Scaling puts the features on comparable ranges, so the calculated distances better reflect the differences between the MOFs rather than the units of the descriptors.

### 7.1. Standard scaling and building a first pipeline 

Given that we will now go beyond training a single model, we will build [Pipelines](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), which are objects that can collect a selection of transformations and estimators. This makes it quite easy to apply the same set of operations to different datasets. A simple pipeline might be built as follows 

<img src="https://vitalflux.com/wp-content/uploads/2020/08/ML-Pipeline-Page-2-1024x307.png" alt="Pipeline" width="800"/>



 $\color{DarkBlue}{\textsf{Short Exercise}}$
- Build a pipline that first performs standard scaling and then fits a KRR. Call it `pipe_w_scaling`. 
- Fit it on the training set 
- Make predictions, calculate the errors and make the parity plots

<details>
<summary> <font color='green'>Click here for hints</font></summary>
<ul>
    <li> the <code>fit</code>, <code>predict</code> methods also work for pipelines </li>
</ul>
</details>

In [ ]:
pipe_w_scaling = Pipeline([
    ('scaling', StandardScaler()),
    ('krr', KernelRidge(kernel='rbf'))
])


In [ ]:
pipe_w_scaling.fit(
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

In [ ]:
res_train_scaled = {
    "y true": df_train_stratified[TARGET],
    "y pred": pipe_w_scaling.predict(
        df_train_stratified[FEATURES]
    )
}

res_test_scaled = {
    "y true": df_test_stratified[TARGET],
    "y pred": pipe_w_scaling.predict(
        df_test_stratified[FEATURES]
    )
}

In [ ]:
res_train_scaled["error"] = (
    res_train_scaled["y true"]
    - res_train_scaled["y pred"]
)

res_test_scaled["error"] = (
    res_test_scaled["y true"]
    - res_test_scaled["y pred"]
)

In [ ]:
scaled_krr_results_train = get_regression_metrics(
    pipe_w_scaling,
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

scaled_krr_results_test = get_regression_metrics(
    pipe_w_scaling,
    df_test_stratified[FEATURES],
    df_test_stratified[TARGET]
)

print("Scaled KRR - train:")
print(scaled_krr_results_train)

print("Scaled KRR - test:")
print(scaled_krr_results_test)

In [ ]:
hex_train_scaled = hv.HexTiles(
    res_train_scaled,
    ["y true", "y pred"]
).opts(
    title="Scaled KRR - Training set",
    xlabel="True CO2 uptake (mmol/g)",
    ylabel="Predicted CO2 uptake (mmol/g)"
).hist(
    dimension=["y true", "y pred"]
)

hex_test_scaled = hv.HexTiles(
    res_test_scaled,
    ["y true", "y pred"]
).opts(
    title="Scaled KRR - Test set",
    xlabel="True CO2 uptake (mmol/g)",
    ylabel="Predicted CO2 uptake (mmol/g)"
).hist(
    dimension=["y true", "y pred"]
)

display(hex_train_scaled + hex_test_scaled)

In [ ]:
plt.figure(figsize=(7, 5))

plt.hist(
    res_train_scaled["error"],
    bins=50,
    alpha=0.6,
    label="Train"
)

plt.hist(
    res_test_scaled["error"],
    bins=50,
    alpha=0.6,
    label="Test"
)

plt.xlabel("Prediction error (mmol CO2/g)")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    res_train_scaled["y true"],
    res_train_scaled["error"],
    alpha=0.3,
    label="Train"
)

plt.scatter(
    res_test_scaled["y true"],
    res_test_scaled["error"],
    alpha=0.3,
    label="Test"
)

plt.axhline(0, linestyle="--")

plt.xlabel("True CO2 uptake (mmol/g)")
plt.ylabel("Error = true - predicted uptake (mmol/g)")
plt.legend()
plt.show()

### Effect of scaling

Standard scaling strongly improves the KRR performance. The test MAE decreases from about **11.7 mmol/g** for the raw KRR to about **1.09 mmol/g** after scaling, and the train/test errors become much closer.

This confirms that scaling is essential for an RBF kernel: once the descriptors are placed on comparable numerical scales, the distances between MOFs become much more meaningful.


### 7.2. Hyperparameter optimization

A key component we did not optimize so far are hyperparameters. Those are parameters of the model that we usually cannot learn from the data but have to fix before we train the model. 
Since we cannot learn those parameters it is not trivial to select them. Hence, what we typically do in practice is to create another set, a "validation set", and use it to test models trained with different hyperparameters.

The most common approach to hyperparameter optimization is to define a grid of all relevant parameters and to search over the grid for the best model performance.

$\color{DarkBlue}{\textsf{Short Exercise}}$

- Think about which parameters you could optimize in the pipeline. Note that your KRR model has two parameters you can optimize. You can also switch off some steps by setting them to `None'.

We can optimize the preprocessing step by comparing different scaling methods, as well as the `alpha` and `gamma` parameters of the KRR model. `Alpha` controls the strength of the regularization, while `gamma` controls how local the RBF kernel is when comparing different materials. We can also test whether the scaling step should be applied or switched off.

- For each parameter you need to define a resonable grid to search over.

A reasonable grid should cover several orders of magnitude for `alpha` and `gamma`, since their optimal values are not known in advance. For the preprocessing step, different options such as no scaling, `MinMaxScaler()` and `StandardScaler()` can be compared.

- Recall, what k-fold cross-validation does. Run the hyperparameter optimization using 5-fold cross-validation (you can adjust the number of folds according to your computational resources/impatience. It turns out at k=10 is the [best tradeoff between variance and bias](https://arxiv.org/abs/1811.12808).  
Tune the hyperparameters until you are statisfied (e.g., until you cannot improve the cross validated error any more)

In 5-fold cross-validation, the training set is divided into five parts. The model is trained on four folds and evaluated on the remaining one. This is repeated five times so that each fold is used once for validation. The average validation performance is then used to compare the different hyperparameter combinations.

- Why don't we use the test set for hyperparameter tuning but instead test on the validation set?

We do not use the test set for hyperparameter tuning because it should remain completely independent from the model development. If we repeatedly use the test performance to choose the best hyperparameters, information from the test set indirectly influences the model and the final evaluation becomes overly optimistic.

Instead, hyperparameters are selected using the validation folds during cross-validation, and the test set is only used once at the end to evaluate how well the final model performs on unseen data.

- Evaluate the model performance by calculating the performance metrics (MAE, MSE, max error) on the training and the test set.

After hyperparameter optimization, the model is evaluated on both the training and test sets using MAE, MSE and maximum error. Comparing the two sets allows us to check whether the optimized model performs well not only on the training data but also on unseen materials.

- **Optional:** Instead of grid search, try to use random search on the same grid (`RandomizedSearchCV`) and fix the number of evaluations (`n_iter`) to a fraction of the number of evaluations of grid search. What do you observe and conclude?

The random search reached the same best cross-validation MAE as the grid search, about 0.924 mmol/g, while evaluating only 12 parameter combinations instead of 36. In this case, random search therefore found an equally good model with only one third of the evaluations. This shows that random search can explore the hyperparameter space more efficiently than an exhaustive grid search, although it does not guarantee that the best combination will always be sampled.

 $\color{DarkRed}{\textsf{Tips}}$
- If you want to see what is happening, set the `verbosity` argument of the `GridSearchCV` object to a higher number.
 
- If you want to speed up the optimization, you can run it in parallel by setting the `n_jobs` argument to the number of workers. If you set it to -1 it will use all available cores. *Using all cores might freeze your computer if you do not have enough memory*
 
- If the optimization is too slow, reduce the number of data points in your set, the number of folds or the grid size. Note that it can also be a feasible strategy to first use a coarser grid and the a finer grid for fine-tuning.

- For grid search, you need to define a parameter grid, which is a dictionary of the following form: 
```(python)
param_grid = {
                    'pipelinestage__parameter': np.logspace(-4,1,10),
                    'pipelinestage': [None, TransformerA(), TransformerB()]
            }
```

- After the search, you can access the best model with `.best_estimator_` and the best parameters with `.best_params_` on the GridSearchCV instance. For example `grid_krr.best_estimator_`

- If you initialize the GridSearchCV instance with `refit=True` it will automatically train the model with all training data (and not only the training folds from cross-validations)

The double underscore (dunder) notation works recursively and specifies the parameters for any pipeline stage. 
For example, `ovasvm__estimator__cls__C` would specifiy the `C` parameter of the estimator in the one-versus-rest classifier `ovasvm`. 

You can print all parameters of the pipeline using `print(sorted(pipeline.get_params().keys()))`

<div class="alert alert-block alert-warning">
Be aware that tight grids will drastically increase the number of experiments you will run! In some cases, it can be useful to perform the optimization in steps, i.e., first use a coarse grid and then refine in interesting regions. 
Alternatively, there are approached like <a href="https://www.jmlr.org/papers/volume18/16-558/16-558.pdf"> hyperband <a> that dynamically adjust the number of data points.
</div>

<details>
<summary> <font color='green'>Click here for hints about pipelines and grid search</font></summary>
<ul>
    <li> You can use the <code>np.logspace</code> function to generate a grid for values that you want to vary on a logarithmic scale </li>
    <li> There are two hyperparameters for KRR: the regularization strength <code>alpha</code> and the Gaussian width  <code>gamma</code> </li>
    <li> For the regularization strength, values between 1 and 1e-3 can be reasonable. For gamma you can use the median heuristic, gamma = 1 / median, or values between 1e-3 and 1e3</li>
</ul>
</details>

In [ ]:
print(sorted(pipe_w_scaling.get_params().keys()))

In [ ]:
param_grid = {
    'scaling': [
        None,
        MinMaxScaler(),
        StandardScaler()
    ],

    'krr__alpha': [
        1e-3,
        1e-2,
        1e-1,
        1
    ],

    'krr__gamma': [
        1e-4,
        1e-3,
        1e-2
    ]
}

In [ ]:
df_tuning, _ = train_test_split(
    df_train_stratified,
    train_size=3000,
    random_state=RANDOM_SEED,
    stratify=df_train_stratified[TARGET_BINARY]
)

In [ ]:
grid_krr = GridSearchCV(
    pipe_w_scaling,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    verbose=2,
    n_jobs=2,
    refit=True
)

grid_krr.fit(
    df_tuning[FEATURES],
    df_tuning[TARGET]
)

In [ ]:
print("Best parameters:")
print(grid_krr.best_params_)

print("Best cross-validation MAE:")
print(-grid_krr.best_score_)

In [ ]:
best_krr = grid_krr.best_estimator_

best_krr.fit(
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

In [ ]:
# get the performance metrics
optimized_results_train = get_regression_metrics(
    best_krr,
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

optimized_results_test = get_regression_metrics(
    best_krr,
    df_test_stratified[FEATURES],
    df_test_stratified[TARGET]
)

print("Optimized KRR - train:")
print(optimized_results_train)

print("Optimized KRR - test:")
print(optimized_results_test)

In [ ]:
random_krr = RandomizedSearchCV(
    pipe_w_scaling,
    param_distributions=param_grid,
    n_iter=12,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=RANDOM_SEED,
    verbose=2,
    n_jobs=2
)

random_krr.fit(
    df_tuning[FEATURES],
    df_tuning[TARGET]
)

print("Grid search best MAE:", -grid_krr.best_score_)
print("Random search best MAE:", -random_krr.best_score_)

### Hyperparameter optimization results

The optimized KRR reaches a test MAE of about **0.76 mmol/g**, a major improvement over both the raw and simply scaled KRR models.

The optional random search reaches the same best cross-validation MAE as the full grid search (about **0.924 mmol/g**) while evaluating only **12 combinations instead of 36**. In this case, random search therefore finds an equally good solution with only one third of the parameter evaluations.


<details>
<summary> <font color='green'>Click here for some more information about hyperparameter optimization</font></summary>
Grid search is not the most efficient way to perform hyperparamter optimization. Even <a href="http://jmlr.csail.mit.edu/papers/volume13/bergstra12a/bergstra12a.pdf">random search was shown to be more efficient</a>. Really efficient though are Bayesian optimization approaches like <a href='https://papers.nips.cc/paper/4443-algorithms-for-hyper-parameter-optimization.pdf)'>TPE</a>. This is implemented in the hyperopt library, which is also installed in your conda environment.
</details>

<details>
<summary> <font color='green'>Click here for hyperparameter optimization with hyperopt (advanded and optional outlook)</font></summary>
    
<b>Import the tools we need</b>
<code>
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, mix, rand, anneal, space_eval
from functools import partial
</code>    

<b>Define the grid</b>
<code>
param_hyperopt = {
    "krr__alpha": hp.loguniform("krr__alpha", np.log(0.001), np.log(10)),
    "krr__gamma": hp.loguniform("krr__gamma", np.log(0.001), np.log(10)),
}
</code> 

<b>Define the objective function</b>
<code>
def objective_function(params):
    pipe.set_params(
        **{
            "krr__alpha": params["krr__alpha"],
            "krr__gamma": params["krr__gamma"],
        }
    )
    score = cross_val_score(
        pipe, X_train, y_train, cv=10, scoring="neg_mean_absolute_error"
    ).mean()
    return {"loss": -score, "status": STATUS_OK} 
</code>

<b>We will use a search in which we mix random search, annealing and tpe</b>
<code>
trials = Trials()
mix_search = partial(
   mix.suggest,
   p_suggest=[(0.15, rand.suggest), (0.15, anneal.suggest), (0.7, tpe.suggest)],
)
</code>

<b>Now, we can minimize the objective function.</b>
<code>
best_param = fmin(
        objective_function,
        param_hyperopt,
        algo=mix_search,
        max_evals=MAX_EVALES,
        trials=trials,
        rstate=np.random.RandomState(RANDOM_SEED),
    )
</code>

</details>

## 8. Feature Engineering 

Finally, we would like to remove features with low variance. This can be done by setting a variance threshold.

$\color{DarkBlue}{\textsf{Short Question}}$
    
- What is the reasoning behind doing this? 

Features with very low variance change very little from one material to another, so they usually contain little useful information for distinguishing the MOFs. Removing them can reduce the dimensionality of the dataset, simplify the model and decrease the computational cost without losing much information.

- When might it go wrong and why?

Low variance does not always mean that a feature is unimportant. For example, a rare feature may vary only for a small number of materials but still have a strong effect on the target. The variance also depends on the numerical scale of the feature, so choosing a threshold that is too high may remove useful information.

 $\color{DarkBlue}{\textsf{Short Exercise}}$
- Add a variance threshold to the pipeline (select the correct function argument)
- Use random search for hyperparameter optimization, retrain the pipeline, and calculate the performance metrics (max error, MAE, MSE) on the training and test set

In [ ]:
# Define the pipeline
pipe_variance_threshold = Pipeline([
    ('variance_threshold', VarianceThreshold()),
    ('scaling', StandardScaler()),
    ('krr', KernelRidge(kernel='rbf'))
])

In [ ]:
param_grid_variance_threshold = {
    'variance_threshold__threshold': [
        0.0,
        1e-5,
        1e-4,
        1e-3,
        1e-2
    ],

    'scaling': [
        MinMaxScaler(),
        StandardScaler()
    ],

    'krr__alpha': [
        1e-4,
        3e-4,
        1e-3,
        3e-3
    ],

    'krr__gamma': [
        3e-4,
        1e-3,
        3e-3
    ]
}

In [ ]:
# Fit the pipeline and run the evaluation
random_variance_threshold = RandomizedSearchCV(
    pipe_variance_threshold,
    param_distributions=param_grid_variance_threshold,
    n_iter=20,
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=RANDOM_SEED,
    verbose=2,
    n_jobs=2,
    refit=True
)

random_variance_threshold.fit(
    df_tuning[FEATURES],
    df_tuning[TARGET]
)

print("Best parameters:")
print(random_variance_threshold.best_params_)

print("Best cross-validation MAE:")
print(-random_variance_threshold.best_score_)

In [ ]:
best_variance_model = random_variance_threshold.best_estimator_

best_variance_model.fit(
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

In [ ]:
variance_results_train = get_regression_metrics(
    best_variance_model,
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

variance_results_test = get_regression_metrics(
    best_variance_model,
    df_test_stratified[FEATURES],
    df_test_stratified[TARGET]
)

print("Variance-threshold KRR - train:")
print(variance_results_train)

print("Variance-threshold KRR - test:")
print(variance_results_test)

In [ ]:
selector = best_variance_model.named_steps['variance_threshold']

n_features_before = len(FEATURES)
n_features_after = selector.get_support().sum()

print("Features before:", n_features_before)
print("Features after:", n_features_after)
print("Features removed:", n_features_before - n_features_after)

### Feature-engineering result

The variance threshold removes **26 of the 164 features**, leaving **138 descriptors**. The resulting model gives a test MAE of about **0.739 mmol/g**, slightly better than the optimized KRR without variance filtering.

This shows that removing nearly constant descriptors can simplify the input space without losing useful predictive information.


$\color{DarkBlue}{\textsf{Short Exercise (optional)}}$
- replace the variance threshold with a model-based feature selection 
`('feature_selection', SelectFromModel(LinearSVC(penalty="l1")))` or [any feature selection method that you would like to try](https://scikit-learn.org/stable/modules/feature_selection.html)

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Lasso

pipe_model_selection = Pipeline([
    ('scaling', StandardScaler()),
    ('feature_selection', SelectFromModel(
        Lasso(max_iter=10000)
    )),
    ('krr', KernelRidge(kernel='rbf'))
])

param_grid_model_selection = {
    'feature_selection__estimator__alpha': [
        1e-4,
        1e-3,
        1e-2,
        1e-1
    ],

    'feature_selection__threshold': [
        'mean',
        'median'
    ],

    'krr__alpha': [
        1e-4,
        3e-4,
        1e-3,
        3e-3
    ],

    'krr__gamma': [
        3e-4,
        1e-3,
        3e-3
    ]
}

random_model_selection = RandomizedSearchCV(
    pipe_model_selection,
    param_distributions=param_grid_model_selection,
    n_iter=20,
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=RANDOM_SEED,
    verbose=2,
    n_jobs=2,
    refit=True
)

random_model_selection.fit(
    df_tuning[FEATURES],
    df_tuning[TARGET]
)

print("Best parameters:")
print(random_model_selection.best_params_)

print("Best cross-validation MAE:")
print(-random_model_selection.best_score_)

best_model_selection = random_model_selection.best_estimator_

best_model_selection.fit(
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

model_selection_train = get_regression_metrics(
    best_model_selection,
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

model_selection_test = get_regression_metrics(
    best_model_selection,
    df_test_stratified[FEATURES],
    df_test_stratified[TARGET]
)

print("Model-based feature selection - train:")
print(model_selection_train)

print("Model-based feature selection - test:")
print(model_selection_test)

selector = best_model_selection.named_steps['feature_selection']

n_features_before = len(FEATURES)
n_features_after = selector.get_support().sum()

print("Features before:", n_features_before)
print("Features after:", n_features_after)
print("Features removed:", n_features_before - n_features_after)


The model-based feature selection is much more aggressive than the variance threshold, reducing the number of features from 164 to only 40. Despite this large reduction, the test MAE remains relatively low, increasing only from about 0.74 to 0.79 mmol/g.

However, the overall performance is slightly worse and the maximum error increases strongly. One possible reason is that the Lasso model used for feature selection mainly identifies features that are useful for linear relationships, while the final KRR model can also use nonlinear relationships. Some features removed by Lasso may therefore still contain useful information for KRR.

This approach gives a much simpler model, but in our case the variance-threshold method provides better predictive performance.

## 9. Saving the model

Now, that we spent so much time in optimizing our model, we do not want to loose it. 

 $\color{DarkBlue}{\textsf{Short Exercise}}$
- use the [joblib library](https://scikit-learn.org/stable/modules/model_persistence.html) to save your model
- make sure you can load it again


In [ ]:
# Save the complete optimized pipeline
filename = "best_krr_model.joblib"
joblib.dump(best_variance_model, filename)


In [ ]:
# Try to load it again
model_loaded = joblib.load(filename)

In [ ]:
# Verify that the reloaded pipeline gives the same predictions
pred_before = best_variance_model.predict(df_test_stratified[FEATURES])
pred_after = model_loaded.predict(df_test_stratified[FEATURES])

print("Predictions identical after reloading:", np.allclose(pred_before, pred_after))


The complete optimized pipeline is saved with `joblib`, including the variance filtering, scaling and KRR model. Saving the whole pipeline ensures that the same preprocessing is applied when the model is reused. The prediction check above can be used to confirm that the reloaded model gives the same predictions as the original one.


## 10. Influence of Regularization

 $\color{DarkBlue}{\textsf{Short Exercise}}$
- what happens if you set $\alpha$  to a really small or too large value? Why is this the case explain what the parameter means using the equation derived in the lectures?

In KRR, the regularization parameter $\alpha$ appears in the solution

$$
\mathbf{c} = (K + \alpha I)^{-1}\mathbf{y}.
$$

A very small $\alpha$ gives little regularization, allowing the model to fit the training data very closely. This can lead to overfitting and poor generalization to new materials.

A very large $\alpha$ gives much stronger regularization and reduces the influence of the fitted coefficients. The model then becomes too constrained and may underfit the data. We therefore expect an intermediate value of $\alpha$ to give the best balance between fitting the training data and generalizing to unseen data.

 To test this, fix this value in one of your pipelines, retrain the models (re-optimizing the other hyperparameters) and rerun the performance evaluation.

<details>
<summary> <font color='green'>Click here for hints</font></summary>
<ul>
    <li> Check the derivation for ridge regression and KRR in the notes. </li>
    <li> Also remember the loss landscapes we discussed in the lectures about LASSO. </li>
</ul>
</details>

In [ ]:
# Test with small value of alpha for KRR
param_grid_small_alpha = {
    'variance_threshold__threshold': [
        0.0,
        1e-5,
        1e-4,
        1e-3
    ],

    'scaling': [
        StandardScaler()
    ],

    'krr__alpha': [
        1e-8
    ],

    'krr__gamma': [
        3e-4,
        1e-3,
        3e-3
    ]
}

search_small_alpha = RandomizedSearchCV(
    pipe_variance_threshold,
    param_distributions=param_grid_small_alpha,
    n_iter=10,
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=RANDOM_SEED,
    verbose=2,
    n_jobs=2,
    refit=True
)

search_small_alpha.fit(
    df_tuning[FEATURES],
    df_tuning[TARGET]
)

model_small_alpha = search_small_alpha.best_estimator_

model_small_alpha.fit(
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)
small_alpha_train = get_regression_metrics(
    model_small_alpha,
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

small_alpha_test = get_regression_metrics(
    model_small_alpha,
    df_test_stratified[FEATURES],
    df_test_stratified[TARGET]
)

print("Small alpha - train:")
print(small_alpha_train)

print("Small alpha - test:")
print(small_alpha_test)




In [ ]:
# Test with large value of alpha for KRR
param_grid_large_alpha = {
    'variance_threshold__threshold': [
        0.0,
        1e-5,
        1e-4,
        1e-3
    ],

    'scaling': [
        StandardScaler()
    ],

    'krr__alpha': [
        100
    ],

    'krr__gamma': [
        3e-4,
        1e-3,
        3e-3
    ]
}

search_large_alpha = RandomizedSearchCV(
    pipe_variance_threshold,
    param_distributions=param_grid_large_alpha,
    n_iter=10,
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=RANDOM_SEED,
    verbose=2,
    n_jobs=2,
    refit=True
)

search_large_alpha.fit(
    df_tuning[FEATURES],
    df_tuning[TARGET]
)

model_large_alpha = search_large_alpha.best_estimator_

model_large_alpha.fit(
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

large_alpha_train = get_regression_metrics(
    model_large_alpha,
    df_train_stratified[FEATURES],
    df_train_stratified[TARGET]
)

large_alpha_test = get_regression_metrics(
    model_large_alpha,
    df_test_stratified[FEATURES],
    df_test_stratified[TARGET]
)

print("Large alpha - train:")
print(large_alpha_train)

print("Large alpha - test:")
print(large_alpha_test)

In [ ]:
print("Optimized alpha:")
print(variance_results_train)
print(variance_results_test)

print("\nVery small alpha:")
print(small_alpha_train)
print(small_alpha_test)

print("\nVery large alpha:")
print(large_alpha_train)
print(large_alpha_test)

The effect of $\alpha$ is clearly visible in the results. With a very small $\alpha$, the training MAE becomes almost zero, but the test MAE increases strongly and some prediction errors become extremely large. This shows that the model is overfitting the training data.

With a very large $\alpha$, both the training and test errors increase to similar values. The model is then too strongly regularized and cannot fit the data properly, which corresponds to underfitting.

The optimized value of $\alpha$ gives the best compromise between fitting the training data and generalizing to unseen materials.

## 11. Interpreting the model 

Now, that our model performs decently, we would like to know which features are mainly responsible for this, i.e. how the model performs its reasoning. 

One method to do so is the [permutation feature importance technique](https://christophm.github.io/interpretable-ml-book/feature-importance.html).

$\color{DarkBlue}{\textsf{Short question}}$

We use both descriptors that encode the pore geometry (density, pore diameters, surface areas) as well as some that describe the chemistry of the MOF (the RACs). 
- Would you expect the relative importance of these features to be different for prediction of gas adsorption at high vs low gas pressure?

Yes, we expect the relative importance of the descriptors to change with pressure. At low pressure, adsorption is more strongly controlled by specific interactions between the gas molecules and the MOF, so chemical descriptors such as the RACs should become more important.

At high pressure, the pores become increasingly filled and the total amount of available space becomes more important. Geometrical descriptors such as pore volume, surface area, pore size and density are therefore expected to play a larger role.

<details>
<summary> <font color='green'>Click here for a hint</font></summary>
<ul>
    <li> <a href="https://pubs.acs.org/doi/abs/10.1021/acs.chemmater.8b02257">An article from Diego et al.</a> (10.1021/acs.chemmater.8b02257) gives some hints.</li>
</ul>
</details>

 $\color{DarkBlue}{\textsf{Short Exercise}}$
- Complete the function `_calculate_permutation_scores` (which we took from the `sklearn` package) and which is needed to calculate the permutation feature importance using the `permutation_importance` function. 

In [ ]:
def _calculate_permutation_scores(estimator, X, y, col_idx, random_state,
                                  n_repeats, scorer):
    """Calculate score when `col_idx` is permuted. Based on the sklearn implementation

    estimator: sklearn estimator object
    X: pd.Dataframe or np.array
    y: pd.Dataframe or np.array
    col_idx: int
    random_state: int
    n_repeats: int
    scorer: function that takes model, X and y_true as arguments
    """
    random_state = check_random_state(random_state)

    X_permuted = X.copy()
    scores = np.zeros(n_repeats)
    # get the indices
    shuffling_idx = np.arange(X.shape[0])
    for n_round in range(n_repeats):
        # FILL BELOW HERE
        # shuffle them (fill in what you want to shuffle)
        random_state.shuffle(shuffling_idx)

        # Deal with dataframes
        if hasattr(X_permuted, "iloc"):
            # .iloc selects the indices from a dataframe and you give it [row, column]
            col = X_permuted.iloc[shuffling_idx, col_idx]
            col.index = X_permuted.index
            X_permuted.iloc[:, col_idx] = col

        # Deal with numpy arrays
        else:
            # FILL BELOW HERE
            # array indexing is [row, column]
            X_permuted[:, col_idx] = X_permuted[shuffling_idx, col_idx]

        # Get the scores
        feature_score = scorer(estimator, X_permuted, y)

        # record the scores in array
        scores[n_round] = feature_score

    return scores

Nothing to change in the function below, it just call the `_calculate_permutation_scores` function you just completed. 

In [ ]:
def permutation_importance(
    estimator,
    X,
    y,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    n_jobs=2,
    random_state=None,
):
    """Permutation importance for feature evaluation
    estimator : object
        An estimator that has already been :term:`fitted` and is compatible
        with :term:`scorer`.
    X : ndarray or DataFrame, shape (n_samples, n_features)
        Data on which permutation importance will be computed.
    y : array-like or None, shape (n_samples, ) or (n_samples, n_classes)
        Targets for supervised or `None` for unsupervised.
    scoring : string, callable or None, default=None
        Scorer to use. It can be a single
        string (see :ref:`scoring_parameter`) or a callable (see
        :ref:`scoring`). If None, the estimator's default scorer is used.
    n_repeats : int, default=5
        Number of times to permute a feature.
    n_jobs : int or None, default=2
        The number of jobs to use for the computation.
        `None` means 1 unless in a :obj:`joblib.parallel_backend` context.
        `-1` means using all processors. See :term:`Glossary <n_jobs>`
        for more details.
    random_state : int, RandomState instance, or None, default=None
        Pseudo-random number generator to control the permutations of each
        feature. See :term:`random_state`.
    """
    # Deal with dataframes
    if not hasattr(X, "iloc"):
        X = check_array(X, force_all_finite="allow-nan", dtype=None)

    # Precompute random seed from the random state to be used
    # to get a fresh independent RandomState instance for each
    # parallel call to _calculate_permutation_scores, irrespective of
    # the fact that variables are shared or not depending on the active
    # joblib backend (sequential, thread-based or process-based).
    random_state = check_random_state(random_state)
    random_seed = random_state.randint(np.iinfo(np.int32).max + 1)

    # Determine scorer from user options.
    scorer = check_scoring(estimator, scoring=scoring)
    # get the performance score on the unpermuted data
    baseline_score = scorer(estimator, X, y)

    # run the permuted evaluations in parallel for each column
    scores = Parallel(n_jobs=n_jobs)(
        delayed(_calculate_permutation_scores)(
            estimator, X, y, col_idx, random_seed, n_repeats, scorer
        )
        for col_idx in range(X.shape[1])
    )

    # get difference two
    importances = baseline_score - np.array(scores)

    # return the results (dictionary)
    return Bunch(
        importances_mean=np.mean(importances, axis=1),
        importances_std=np.std(importances, axis=1),
        importances=importances,
    )


 $\color{DarkBlue}{\textsf{Short Exercise}}$
- Use your function to find the five most important features.
- Which are they? Did you expect this result?

In [ ]:
permutation_results = permutation_importance(
    best_variance_model,
    df_test_stratified[FEATURES],
    df_test_stratified[TARGET],
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    n_jobs=2,
    random_state=RANDOM_SEED
)

In [ ]:
top5_indices = np.argsort(
    permutation_results.importances_mean
)[-5:][::-1]

top5_features = np.array(FEATURES)[top5_indices]

top5_importances = (
    permutation_results.importances_mean[top5_indices]
)

print("Five most important features:")

for feature, importance in zip(
    top5_features,
    top5_importances
):
    print(feature, importance)

### Most important features

The five most important features are:

1. `sum-f-lig-S-2`
2. `total_SA_gravimetric`
3. `sum-f-lig-S-0`
4. `total_POV_gravimetric`
5. `total_POV_volumetric`

Three of the five are geometrical descriptors related to surface area or pore volume, which is consistent with the expectation that pore filling is important for high-pressure adsorption. Two linker RAC descriptors also appear among the most important features, showing that framework chemistry still contributes to the prediction.

Permutation importance should nevertheless be interpreted carefully when features are strongly correlated, because correlated descriptors can partly replace one another.


In [ ]:
permutation_results["features"] = FEATURES

bars = hv.Bars(
    permutation_results,
    "features",
    ["importances_mean", "importances_std"]
).sort(
    "importances_mean",
    reverse=True
)

errors = hv.ErrorBars(
    permutation_results,
    "features",
    vdims=["importances_mean", "importances_std"]
).sort(
    "importances_mean",
    reverse=True
)

bars * errors

<details>
<summary> <font color='green'>Click here for hints</font></summary>
<ul>
    <li> To get the top <emph>n</emph> indices of an array <code>a</code>, you can use <code>np.argsort(a)[-n:]</code></li>
    <li> Get the feature names from the <code>FEATURES</code> list </li> 
    <li> combined this might look like <code>np.array(FEATURES)[np.argsort(a)[-n:]]</code></li>
</ul>
</details>

<details>
<summary> <font color='green'>Click here for more information on model interpretation</font></summary>
The permutation feature importance technique is not a silver bullet, e.g. there are issues with correlated features.
However, it is likely <a href='https://explained.ai/rf-importance/'>a better choice than feature importance, like impurity decrease, derived from random forests</a>).
</details>

## Final summary

The workflow shows how strongly preprocessing affects model quality. The raw KRR performs poorly on unseen MOFs, while scaling reduces the test MAE from about **11.7** to **1.09 mmol/g**. Hyperparameter optimization further reduces it to about **0.76 mmol/g**, and variance filtering gives the best main-model result at about **0.74 mmol/g** while removing 26 low-variance descriptors.

The final model therefore combines **variance filtering, standard scaling and RBF kernel ridge regression**. Its feature-importance analysis indicates that both pore geometry and framework chemistry contribute to high-pressure CO₂ uptake.
